# Manual Labeler Analysis

This notebook analyzes the agreement between judge ratings and manual labels.

In [ ]:
import json

# Load the judge ratings data
judge_data = []
with open('results/training_data_hidden_prompt_ratings_gpt_4_1_short_selection.jsonl', 'r') as f:
    for line in f:
        judge_data.append(json.loads(line))

# Load the manual labels data
manual_data = []
with open('results/training_generations_hidden_prompt_gpt_4-1-mini_short_manual_labels.jsonl', 'r') as f:
    for line in f:
        manual_data.append(json.loads(line))

print(f"Loaded {len(judge_data)} judge ratings")
print(f"Loaded {len(manual_data)} manual labels")

In [ ]:
# Create a dictionary for quick lookup of judge ratings by (doc_index, scenario)
judge_dict = {}
for entry in judge_data:
    key = (entry['doc_index'], entry['scenario'])
    judge_dict[key] = entry

print(f"Judge ratings cover {len(judge_dict)} unique (doc_index, scenario) combinations")

In [ ]:
# Find matching entries and compute agreement
matches = []
agreements = 0
disagreements = 0

for manual_entry in manual_data:
    key = (manual_entry['doc_index'], manual_entry['scenario'])
    
    # if 'no_intervention' in [manual_entry['accepted_type'], manual_entry['rejected_type']]:
    #     continue

    if key in judge_dict:
        judge_entry = judge_dict[key]
        
        # Determine which agent won in judge data (accepted_agent)
        judge_winner = judge_entry['accepted_agent']
        
        # Determine which agent won in manual data
        manual_winner = manual_entry['accepted_agent']
        
        # Check if they agree
        agree = (judge_winner == manual_winner)
        
        if agree:
            agreements += 1
        else:
            disagreements += 1
        
        matches.append({
            'doc_index': manual_entry['doc_index'],
            'scenario': manual_entry['scenario'],
            'judge_winner': judge_winner,
            'manual_winner': manual_winner,
            'agree': agree,
            'accepted_type': judge_entry['accepted_type'],
            'manual_accepted_type': manual_entry['accepted_type']
        })
    else:
        print(f"Manual label (doc {manual_entry['doc_index']}, scenario {manual_entry['scenario']}) not found in judge data")

print(f"\nFound {len(matches)} matching (doc_index, scenario) combinations")
print(f"Agreements: {agreements}")
print(f"Disagreements: {disagreements}")
print(f"Agreement ratio: {agreements / len(matches):.2%}")

In [ ]:
# Show some examples of agreements and disagreements
print("\n=== Examples of AGREEMENTS ===")
for match in matches[:5]:
    if match['agree']:
        print(f"Doc {match['doc_index']}, Scenario {match['scenario']}: Both chose agent {match['judge_winner']} ({match['accepted_type']})")

print("\n=== Examples of DISAGREEMENTS ===")
for match in matches:
    if not match['agree']:
        print(f"Doc {match['doc_index']}, Scenario {match['scenario']}: Judge chose agent {match['judge_winner']}, Manual chose agent {match['manual_winner']}")
        print(f"  Judge type: {match['accepted_type']}, Manual type: {match['manual_accepted_type']}")

In [ ]:
# Breakdown by JUDGE intervention type
from collections import defaultdict

type_agreement = defaultdict(lambda: {'agree': 0, 'disagree': 0})

for match in matches:
    intervention_type = match['accepted_type']
    if match['agree']:
        type_agreement[intervention_type]['agree'] += 1
    else:
        type_agreement[intervention_type]['disagree'] += 1

print("\n=== Agreement by Intervention Type ===")
for intervention_type, counts in sorted(type_agreement.items()):
    total = counts['agree'] + counts['disagree']
    ratio = counts['agree'] / total if total > 0 else 0
    print(f"{intervention_type}: {counts['agree']}/{total} ({ratio:.2%})")

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

# Create a mapping of (manual_type, judge_type) combinations and their counts
combination_counts = Counter()

for match in matches:
    manual_type = match['manual_accepted_type']
    judge_type = match['accepted_type']
    combination_counts[(manual_type, judge_type)] += 1

# Define the desired order for intervention types (always show all)
intervention_order = ['no_intervention', 'socratic', 'compromise']

# Use all intervention types for both axes
manual_types = intervention_order
judge_types = intervention_order

# Create numerical mappings for categorical data
manual_type_to_num = {t: i for i, t in enumerate(manual_types)}
judge_type_to_num = {t: i for i, t in enumerate(judge_types)}

# Prepare x, y, and sizes for bubble chart
x_coords = []
y_coords = []
sizes = []
labels = []

for (manual_type, judge_type), count in combination_counts.items():
    x_coords.append(manual_type_to_num[manual_type])
    y_coords.append(judge_type_to_num[judge_type])
    sizes.append(count * 100)  # Scale up for visibility
    labels.append(count)

# Create the bubble chart
plt.figure(figsize=(10, 8))
scatter = plt.scatter(x_coords, y_coords, s=sizes, alpha=0.6, c=sizes, cmap='viridis', edgecolors='black', linewidth=1.5)

# Add count labels on each bubble
for i, (x, y, label) in enumerate(zip(x_coords, y_coords, labels)):
    plt.text(x, y, str(label), ha='center', va='center', fontsize=10, fontweight='bold')

# Set axis labels and ticks
plt.xlabel('Human Labeler - Selected Intervention Type', fontsize=12, fontweight='bold')
plt.ylabel('LLM-as-a-Judge - Selected Intervention Type', fontsize=12, fontweight='bold')
# plt.title('Selection Comparison Between Human Labeler and LLM-as-a-Judge', fontsize=14, fontweight='bold')

plt.xticks(range(len(manual_types)), manual_types, rotation=45, ha='right')
plt.yticks(range(len(judge_types)), judge_types)

# Add grid for better readability
plt.grid(True, alpha=0.3, linestyle='--')

# # Add colorbar to show scale
# cbar = plt.colorbar(scatter, label='Number of Selections')

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\n=== Bubble Chart Summary ===")
print(f"Total combinations: {len(combination_counts)}")
print(f"\nMost common combinations:")
for (manual_type, judge_type), count in combination_counts.most_common(5):
    print(f"  Manual: {manual_type}, Judge: {judge_type} → {count} occurrences")

In [ ]:
# Analyze selection combinations by intervention number (agent 1 vs agent 2)
from collections import Counter
import matplotlib.pyplot as plt

# Create a mapping of (manual_agent, judge_agent) combinations and their counts
agent_combination_counts = Counter()

for match in matches:
    manual_agent = match['manual_winner']
    judge_agent = match['judge_winner']
    agent_combination_counts[(manual_agent, judge_agent)] += 1

# Define agent options (intervention numbers)
agent_options = [1, 2]

# Create numerical mappings
manual_agent_to_num = {a: i for i, a in enumerate(agent_options)}
judge_agent_to_num = {a: i for i, a in enumerate(agent_options)}

# Prepare data for bubble chart
x_coords = []
y_coords = []
sizes = []
labels = []

for (manual_agent, judge_agent), count in agent_combination_counts.items():
    x_coords.append(manual_agent_to_num[manual_agent])
    y_coords.append(judge_agent_to_num[judge_agent])
    sizes.append(count * 200)  # Scale up for visibility
    labels.append(count)

# Create the bubble chart
plt.figure(figsize=(8, 6))
scatter = plt.scatter(x_coords, y_coords, s=sizes, alpha=0.6, c=sizes, cmap='plasma', edgecolors='black', linewidth=1.5)

# Add count labels on each bubble
for i, (x, y, label) in enumerate(zip(x_coords, y_coords, labels)):
    plt.text(x, y, str(label), ha='center', va='center', fontsize=12, fontweight='bold', color='white')

# Set axis labels and ticks
plt.xlabel('Human Labeler - Selected Agent', fontsize=12, fontweight='bold')
plt.ylabel('LLM-as-a-Judge - Selected Agent', fontsize=12, fontweight='bold')

plt.xticks(range(len(agent_options)), [f'Agent {a}' for a in agent_options])
plt.yticks(range(len(agent_options)), [f'Agent {a}' for a in agent_options])

# Add grid for better readability
plt.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\n=== Agent Selection Combination Summary ===")
print(f"Total combinations: {len(agent_combination_counts)}")
print(f"\nAll combinations:")
for (manual_agent, judge_agent), count in sorted(agent_combination_counts.items()):
    agreement = "✓ AGREE" if manual_agent == judge_agent else "✗ DISAGREE"
    print(f"  Manual: Agent {manual_agent}, Judge: Agent {judge_agent} → {count} occurrences ({agreement})")

# Calculate diagonal (agreement) vs off-diagonal (disagreement)
diagonal_sum = sum(count for (m, j), count in agent_combination_counts.items() if m == j)
off_diagonal_sum = sum(count for (m, j), count in agent_combination_counts.items() if m != j)
print(f"\nAgreements (diagonal): {diagonal_sum}")
print(f"Disagreements (off-diagonal): {off_diagonal_sum}")
print(f"Agreement rate: {diagonal_sum / (diagonal_sum + off_diagonal_sum):.2%}")